In [1]:
import torch
from mmasim_kernels.nv_ptx.rtx_blackwell import mma_kernels

torch.manual_seed(0)
HMMA_F32 = mma_kernels["m16n8k16.f32.f16.f16.f32"]
HMMA_F16 = mma_kernels["m16n8k16.f16.f16.f16.f16"]

In [2]:
bsz = 100
A = (10 * torch.randn(bsz, 128, 128, device='cuda:0')).to(torch.float16)
B = (10 * torch.randn(bsz, 128, 128, device='cuda:0')).to(torch.float16)
A, B

(tensor([[[-9.2500e+00, -4.2539e+00, -2.6438e+01,  ..., -2.1289e+00,
           -3.3145e+00, -2.0234e+00],
          [-1.1453e+01, -5.7148e+00, -6.5117e+00,  ...,  1.3414e+01,
            3.3156e+01, -8.4688e+00],
          [ 2.3730e+00,  1.0883e+01, -2.4980e+00,  ...,  2.4297e+00,
            1.7021e+00, -2.1113e+00],
          ...,
          [ 4.5703e+00, -5.8516e+00, -6.3438e+00,  ..., -4.8633e+00,
           -7.6211e+00,  8.6016e+00],
          [-5.6602e+00,  8.8359e+00,  3.9258e-01,  ...,  3.5664e+00,
            6.9844e+00, -8.5742e-01],
          [ 5.4541e-01, -1.7812e+00, -3.3242e+00,  ..., -1.3154e+00,
           -4.2070e+00, -2.9547e+01]],
 
         [[ 1.3545e+00,  2.5352e+00,  9.0859e+00,  ...,  6.6250e+00,
           -1.1805e+01, -5.6133e+00],
          [-2.1387e+00, -2.1172e+00,  1.6672e+01,  ..., -2.3047e-01,
            7.5039e+00,  1.6047e+01],
          [-1.5602e+01, -3.2188e+00,  7.7266e+00,  ..., -3.6836e+00,
            1.2875e+01,  2.1465e+00],
          ...,
    

In [3]:
D_HMMA_F16 = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
D_HMMA_F32 = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_real = A.double() @ B.double()
for t in range(bsz):
    for i in range(0, 128, 16):
        for j in range(0, 128, 8):
            for k in range(0, 128, 16):
                D_HMMA_F32[t, i:i+16, j:j+8] = HMMA_F32(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_HMMA_F32[t, i:i+16, j:j+8])
                D_HMMA_F16[t, i:i+16, j:j+8] = HMMA_F16(A[t, i:i+16, k:k+16], B[t, k:k+16, j:j+8], D_HMMA_F16[t, i:i+16, j:j+8])

In [4]:
print("HMMA.F16 MSE:", (D_real - D_HMMA_F16).square().mean().item())
print("HMMA.F32 MSE:", (D_real - D_HMMA_F32).square().mean().item())
print("HMMA.F32 + Convert to F16 MSE:", (D_real - D_HMMA_F32.half()).square().mean().item())

HMMA.F16 MSE: 0.24717118064910998
HMMA.F32 MSE: 4.48075689392512e-08
HMMA.F32 + Convert to F16 MSE: 0.05512146046856508
